# 导入依赖和定义文件

In [22]:

import json

import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [23]:

# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


# 寻找并保存最优参数

In [24]:

# 定义 LGBM 的目标函数
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'num_leaves': trial.suggest_int('num_leaves', 20, 30),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 定义 CatBoost 的目标函数
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 3),
        'border_count': trial.suggest_categorical('border_count', [32, 64]),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 创建 Optuna 研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

# 获取最优参数
lgbm_best_params = lgbm_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_params = catboost_study.best_params
catboost_best_value = catboost_study.best_value

[I 2025-04-27 15:29:30,284] A new study created in memory with name: no-name-57c9bbdf-56f4-4f18-afd9-e569792a2e43
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:30,341] Trial 0 finished with value: 0.7906843013225991 and parameters: {'n_estimators': 50, 'learning_rate': 0.05799943711170762, 'max_depth': 4, 'num_leaves': 30, 'subsample': 0.9442865290153947, 'colsample_bytree': 0.9449490776480559}. Best is trial 0 with value: 0.7906843013225991.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:30,397] Trial 1 finished with value: 0.7998849913743531 and parameters: {'n_estimators': 100, '

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005451 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:30,516] Trial 4 finished with value: 0.7883841288096607 and parameters: {'n_estimators': 100, 'learning_rate': 0.024985578913782898, 'max_depth': 4, 'num_leaves': 23, 'subsample': 0.8634082477425875, 'colsample_bytree': 0.8671228712485709}. Best is trial 1 with value: 0.7998849913743531.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:30,554] Trial 5 finished with value: 0.7786083956296722 and parameters: {'n_estimators': 50, 'learning_rate': 0.013588919081135142, 'max_depth': 6, 'num_leaves': 28, 'subsample': 0.9961850068985665, 'colsamp

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:30,709] Trial 9 finished with value: 0.7918343875790684 and parameters: {'n_estimators': 100, 'learning_rate': 0.030046136167630914, 'max_depth': 5, 'num_leaves': 22, 'subsample': 0.9473715322554175, 'colsample_bytree': 0.9876499895515961}. Best is trial 1 with value: 0.7998849913743531.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:30,772] Trial 10 finished with value: 0.7901092581943646 and parameters: {'n_estimators': 100, 'learning_rate': 0.047562314495718926, 'max_depth': 5, 'num_leaves': 20, 'subsample': 0.9026856504372743, 'colsa

...

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:32,815] Trial 43 finished with value: 0.7998849913743531 and parameters: {'n_estimators': 100, 'learning_rate': 0.0976991725612695, 'max_depth': 5, 'num_leaves': 28, 'subsample': 0.9127946827763748, 'colsample_bytree': 0.93965383338625}. Best is trial 28 with value: 0.8108108108108109.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:32,867] Trial 44 finished with value: 0.8027602070155262 and parameters: {'n_estimators': 100, 'learning_rate': 0.08991137883183126, 'max_depth': 4, 'num_leaves': 29, 'subsample': 0.987685866667726, 'colsample

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:33,009] Trial 46 finished with value: 0.7855089131684876 and parameters: {'n_estimators': 100, 'learning_rate': 0.011999727035309807, 'max_depth': 6, 'num_leaves': 26, 'subsample': 0.9419037231661875, 'colsample_bytree': 0.8305937394288252}. Best is trial 28 with value: 0.8108108108108109.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 15:29:33,058] Trial 47 finished with value: 0.7935595169637722 and parameters: {'n_estimators': 50, 'learning_rate': 0.07056910766788892, 'max_depth': 5, 'num_leaves': 28, 'subsample': 0.9124380162949511, 'colsa

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

[I 2025-04-27 15:29:33,428] Trial 0 finished with value: 0.7958596894767107 and parameters: {'n_estimators': 100, 'learning_rate': 0.025213510754833763, 'depth': 6, 'l2_leaf_reg': 3, 'border_count': 32}. Best is trial 0 with value: 0.7958596894767107.
[I 2025-04-27 15:29:33,565] Trial 1 finished with value: 0.7860839562967222 and parameters: {'n_estimators': 50, 'learning_rate': 0.027003934374629585, 'depth': 5, 'l2_leaf_reg': 1, 'border_count': 32}. Best is trial 0 with value: 0.7958596894767107.
[I 2025-04-27 15:29:33,791] Trial 2 finished with value: 0.7929844738355377 and parameters: {'n_estimators': 100, 'learning_rate': 0.03348236014698886, 'depth': 5, 'l2_leaf_reg': 2, 'border_count': 32}. Best is trial 0 with value: 0.7958596894767107.
[I 2025-04-27 15:29:34,343] Trial 3 finished with value: 0.7947096032202415 and parameters: {'n_estimators': 100, 'learning_rate': 0.02335376776800077, 'depth': 6, 'l2_leaf_reg': 2, 'border_count': 64}. Best is trial 0 with value: 0.7958596894767

In [25]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)
# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.8108108108108109
CatBoost 最佳得分: 0.8010350776308223


# 训练并保存模型

In [26]:

# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)

lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)

joblib.dump(lgbm_model, 'lgbm_best_model.joblib')
joblib.dump(catboost_model, 'catboost_best_model.joblib')


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000331 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

['catboost_best_model.joblib']

# 模型融合与提交文件

In [ ]:
基学习器已经训练好，现在使用元学习器进行模型融合

In [27]:
# 加载训练好的基模型
lgbm_model = joblib.load('lgbm_best_model.joblib')
catboost_model = joblib.load('catboost_best_model.joblib')

# 检查模型加载是否正确
print(lgbm_model.get_params())
print(catboost_model.get_params())

# 定义生成元特征的函数
def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
    print("Entering generate_meta_features function")  # 调试信息
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    meta_train = np.zeros((X_train.shape[0],))
    meta_valid = np.zeros((X_valid.shape[0],))
    meta_test = np.zeros((X_test.shape[0],))
    
    for train_index, val_index in kf.split(X_train):
        print("Inside KFold loop")  # 调试信息
        X_tr, X_val = X_train[train_index], X_train[val_index]
        y_tr = y_train[train_index]  # 获取当前折的训练目标变量
        
        # 确保 X_tr 和 y_tr 的维度正确
        print(f"X_tr shape: {X_tr.shape}, y_tr shape: {y_tr.shape}")
        
        model.fit(X_tr, y_tr)  # 传递 y_tr 作为目标变量
        meta_train[val_index] = model.predict_proba(X_val)[:, 1]
        print("Completed a fold")  # 调试信息
    
    # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
    print("Refitting model on entire training set")  # 调试信息
    model.fit(X_train, y_train)
    meta_valid = model.predict_proba(X_valid)[:, 1]
    meta_test = model.predict_proba(X_test)[:, 1]
    print("Exiting generate_meta_features function")  # 调试信息
    
    return meta_train, meta_valid, meta_test

# 为 LGBM 和 CatBoost 生成元特征
print("Generating meta features for LGBM")  # 调试信息
lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)
print("Generating meta features for CatBoost")  # 调试信息
catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# 构建元特征矩阵
X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# 定义元模型的目标函数
def meta_objective(trial):
    params = {
        'C': trial.suggest_float('C', 0.01, 10.0, log=True),
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
        'random_state': 0
    }
    model = LogisticRegression(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    accuracy = accuracy_score(y_valid, y_pred)
    print(f"Meta model accuracy: {accuracy:.4f}")  # 调试信息
    return accuracy

# 创建 Optuna 研究对象并进行优化
print("Starting Optuna study for meta model")  # 调试信息
meta_study = optuna.create_study(direction='maximize')
meta_study.optimize(meta_objective, n_trials=50)

# 获取最优参数
meta_best_params = meta_study.best_params
meta_best_value = meta_study.best_value

print(f"元模型最优参数: {meta_best_params}")
print(f"元模型最优准确率: {meta_best_value:.4f}")

# 使用最优参数训练最终的元模型
best_meta_model = LogisticRegression(**meta_best_params, random_state=0)
best_meta_model.fit(X_train_meta, y_train)

# 保存元模型
joblib.dump(best_meta_model, 'best_meta_model.joblib')


{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.817020160011815, 'importance_type': 'split', 'learning_rate': 0.08629075967215366, 'max_depth': 5, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 30, 'objective': None, 'random_state': 0, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.9089701953146146, 'subsample_for_bin': 200000, 'subsample_freq': 0}
{'learning_rate': 0.042140448814244565, 'depth': 6, 'l2_leaf_reg': 2, 'border_count': 32, 'verbose': False, 'n_estimators': 100, 'random_state': 0}
Generating meta features for LGBM
Entering generate_meta_features function
Inside KFold loop
X_tr shape: (5563, 37), y_tr shape: (5563,)
[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000489 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bin

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site

Completed a fold
Inside KFold loop
X_tr shape: (5564, 37), y_tr shape: (5564,)
[LightGBM] [Info] Number of positive: 2815, number of negative: 2749
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000410 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1948
[LightGBM] [Info] Number of data points in the train set: 5564, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.505931 -> initscore=0.023725
[LightGBM] [Info] Start training from score 0.023725
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Completed a fold
Inside KFold loop
X_tr shape: (5563, 37), y_tr shape: (5563,)
Completed a fold
Inside KFold loop
X_tr shape: (5563, 37), y_tr shape: (5563,)
Completed a fold
Inside KFold loop
X_tr shape: (5563, 37), y_tr shape: (5563,)
Completed a fold
Inside KFold loop
X_tr shape: (5564, 37), y_tr shape: (5564,)
Completed a fold
Refitting model on entire training set


[I 2025-04-27 15:29:45,863] A new study created in memory with name: no-name-a0e5193b-f987-42a1-8b0d-6cb3e6243088
[I 2025-04-27 15:29:45,870] Trial 0 finished with value: 0.8004600345025877 and parameters: {'C': 1.7948854957421463, 'solver': 'saga'}. Best is trial 0 with value: 0.8004600345025877.
[I 2025-04-27 15:29:45,874] Trial 1 finished with value: 0.8004600345025877 and parameters: {'C': 3.12622488643589, 'solver': 'newton-cg'}. Best is trial 0 with value: 0.8004600345025877.
[I 2025-04-27 15:29:45,881] Trial 2 finished with value: 0.7998849913743531 and parameters: {'C': 0.2947494988093701, 'solver': 'saga'}. Best is trial 0 with value: 0.8004600345025877.
[I 2025-04-27 15:29:45,887] Trial 3 finished with value: 0.8004600345025877 and parameters: {'C': 2.1207844937956515, 'solver': 'saga'}. Best is trial 0 with value: 0.8004600345025877.
[I 2025-04-27 15:29:45,894] Trial 4 finished with value: 0.8004600345025877 and parameters: {'C': 7.981803281874254, 'solver': 'saga'}. Best is

Exiting generate_meta_features function
Starting Optuna study for meta model
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.7999
Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8010
Meta model accuracy: 0.7999
Meta model accuracy: 0.8010
Meta model accuracy: 0.8005


[I 2025-04-27 15:29:46,065] Trial 27 finished with value: 0.8010350776308223 and parameters: {'C': 0.6821254276192397, 'solver': 'liblinear'}. Best is trial 10 with value: 0.8010350776308223.
[I 2025-04-27 15:29:46,073] Trial 28 finished with value: 0.8010350776308223 and parameters: {'C': 0.41761257401762786, 'solver': 'liblinear'}. Best is trial 10 with value: 0.8010350776308223.
[I 2025-04-27 15:29:46,081] Trial 29 finished with value: 0.8004600345025877 and parameters: {'C': 1.6252964854173997, 'solver': 'liblinear'}. Best is trial 10 with value: 0.8010350776308223.
[I 2025-04-27 15:29:46,093] Trial 30 finished with value: 0.7998849913743531 and parameters: {'C': 0.20610372101376434, 'solver': 'sag'}. Best is trial 10 with value: 0.8010350776308223.
[I 2025-04-27 15:29:46,101] Trial 31 finished with value: 0.8010350776308223 and parameters: {'C': 0.12553002531205423, 'solver': 'liblinear'}. Best is trial 10 with value: 0.8010350776308223.
[I 2025-04-27 15:29:46,109] Trial 32 finish

Meta model accuracy: 0.8010
Meta model accuracy: 0.8010
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.8010
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.7999
Meta model accuracy: 0.7999
Meta model accuracy: 0.8010
Meta model accuracy: 0.8005
Meta model accuracy: 0.8010
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8005
Meta model accuracy: 0.8010
元模型最优参数: {'C': 0.5848701373302152, 'solver': 'liblinear'}
元模型最优准确率: 0.8010


['best_meta_model.joblib']

In [28]:

# 加载元模型
best_meta_model = joblib.load('best_meta_model.joblib')

# 生成预测结果
prob = best_meta_model.predict_proba(X_test_meta)[:, 1]

# 加载测试数据
test_data = pd.read_csv('test.csv')

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': prob > 0.5
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)

In [29]:
# # 加载模型
# lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
# catboost_loaded_model = joblib.load('catboost_best_model.joblib')

# # 对测试数据进行预测，获取概率值
# lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
# catboost_prob = catboost_model.predict_proba(X_test)[:, 1]

# # 简单平均融合
# ensemble_prob = (lgbm_prob + catboost_prob) / 2

# # 根据阈值生成最终预测
# threshold = 0.5
# ensemble_pred = ensemble_prob > threshold

# # 加载测试数据
# test_data = pd.read_csv('test.csv')  # 确保文件路径正确

# # 创建提交文件
# submission = pd.DataFrame({
#     'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
#     'Transported': ensemble_pred
# })

# # 保存为 CSV 文件
# submission.to_csv('submission.csv', index=False)
